# Self-Attention：公式、mask、多头与 Flash 思想

## 学习目标

1. 从张量形状推导 scaled dot-product attention 与多头输出。
2. 从零实现稳定 softmax、causal/padding/segment mask，并验证未来不泄漏。
3. 理解 MHA、MQA、GQA 的头映射与参数/带宽取舍。
4. 用在线 softmax 理解 FlashAttention 为何 exact 且省 IO。
5. 掌握 NaN、全 mask 行、广播轴、transpose 和性能 profile 的常见坑。

$Attention(Q,K,V)=softmax(QK^\top/\sqrt{d_k}+M)V$。本 notebook 只依赖 NumPy。

In [ ]:
import time
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(23)

def stable_softmax(x, mask=None, axis=-1):
    x = np.asarray(x, dtype=np.float64)
    if mask is not None:
        mask = np.asarray(mask, dtype=bool)
        x = np.where(mask, x, -np.inf)
    row_max = np.max(x, axis=axis, keepdims=True)
    all_masked = ~np.isfinite(row_max)
    safe_max = np.where(all_masked, 0.0, row_max)
    shifted = x - safe_max
    e = np.where(np.isfinite(shifted), np.exp(shifted), 0.0)
    denom = e.sum(axis=axis, keepdims=True)
    return np.divide(e, denom, out=np.zeros_like(e), where=denom > 0)

## 1. 单头 scaled dot-product attention

Q 形状 $[L_q,d_k]$，K 为 $[L_k,d_k]$，V 为 $[L_k,d_v]$；score 为 $[L_q,L_k]$，输出为 $[L_q,d_v]$。Softmax 必须沿 key 轴。

In [ ]:
def scaled_dot_attention(q, k, v, visible=None):
    if q.shape[-1] != k.shape[-1] or k.shape[-2] != v.shape[-2]:
        raise ValueError('Q/K head_dim 或 K/V 序列长度不匹配')
    scores = q @ np.swapaxes(k, -1, -2) / np.sqrt(q.shape[-1])
    weights = stable_softmax(scores, visible, axis=-1)
    return weights @ v, weights, scores

q = np.array([[1.0, 0.0], [0.0, 1.0]])
k = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
v = np.array([[10.0, 0.0], [0.0, 20.0], [30.0, 30.0]])
out, weights, scores = scaled_dot_attention(q, k, v)
print('scores', scores.shape, '\n', scores)
print('weights row sums =', weights.sum(axis=-1))
print('output', out.shape, '\n', out)

## 2. 为什么除以 $\sqrt{d_k}$

若 Q/K 分量方差约 1，未缩放点积方差约 $d_k$。缩放后 score 保持 $O(1)$，softmax 不易随维度增大而饱和。它不是余弦：Q/K 没有单位归一化。

In [ ]:
for d in [8, 64, 512]:
    q_sample = rng.normal(size=(4000, d))
    k_sample = rng.normal(size=(4000, d))
    raw = np.sum(q_sample * k_sample, axis=1)
    scaled = raw / np.sqrt(d)
    print(f'd={d:3d}: raw std={raw.std():6.2f}, scaled std={scaled.std():5.2f}')

## 3. Causal、padding 与 segment mask

可见条件可以写成 valid query、valid key、$j\le i$、同 segment 的逻辑与。Position reset 不能替代 segment mask。Bool API 中 True 的含义因框架不同，本实现明确 True=可见。

In [ ]:
def combined_visible(valid, segments=None, causal=True):
    valid = np.asarray(valid, dtype=bool)
    length = valid.size
    visible = valid[:, None] & valid[None, :]
    if causal:
        visible &= np.arange(length)[None, :] <= np.arange(length)[:, None]
    if segments is not None:
        segments = np.asarray(segments)
        visible &= segments[:, None] == segments[None, :]
    return visible

valid = np.array([1, 1, 1, 1, 0], dtype=bool)
segments = np.array([0, 0, 1, 1, -1])
visible = combined_visible(valid, segments)
print(visible.astype(int))

# 全 mask query 行由 stable_softmax 安全输出全零，不产生 NaN。
test_prob = stable_softmax(np.zeros((5, 5)), visible)
print('row sums (PAD query is 0) =', test_prob.sum(axis=-1))

## 4. 未来泄漏差分测试

固定前缀，只改变未来 token，前缀位置输出必须不变。这比只看三角热力图更可靠，也能测试融合 kernel 的 mask 方向。

In [ ]:
x1 = rng.normal(size=(5, 4))
x2 = x1.copy(); x2[3:] = rng.normal(size=(2, 4)) * 100
causal = np.arange(5)[None, :] <= np.arange(5)[:, None]
o1, _, _ = scaled_dot_attention(x1, x1, x1, causal)
o2, _, _ = scaled_dot_attention(x2, x2, x2, causal)
print('prefix max error =', np.max(np.abs(o1[:3] - o2[:3])))
assert np.allclose(o1[:3], o2[:3])

## 5. 从零实现多头投影

输入 $[B,L,D]$ 经 Q/K/V 投影后 reshape 为 $[B,H,L,d_h]$；每头独立 softmax，拼接回 $[B,L,D]$ 再乘 $W_O$。总宽度固定时，多头不是把参数简单乘 H。

In [ ]:
def split_heads(x, heads):
    b, l, d = x.shape
    if d % heads:
        raise ValueError('D 必须能被 heads 整除')
    return x.reshape(b, l, heads, d // heads).transpose(0, 2, 1, 3)

def merge_heads(x):
    b, h, l, dh = x.shape
    return x.transpose(0, 2, 1, 3).copy().reshape(b, l, h * dh)

B, L, D, H = 2, 4, 8, 2
X = rng.normal(size=(B, L, D))
Wq, Wk, Wv, Wo = [rng.normal(scale=0.2, size=(D, D)) for _ in range(4)]
Q, K, V = [split_heads(X @ W, H) for W in (Wq, Wk, Wv)]
mask4d = (np.arange(L)[None, :] <= np.arange(L)[:, None])[None, None, :, :]
head_out, attn, _ = scaled_dot_attention(Q, K, V, mask4d)
Y = merge_heads(head_out) @ Wo
print('Q/K/V:', Q.shape, 'attention:', attn.shape, 'output:', Y.shape)
assert Y.shape == (B, L, D)

## 6. GQA/MQA 的 KV 头映射

MHA 有 $H_{kv}=H_q$，MQA 有 $H_{kv}=1$，GQA 介于两者。若 $H_q=4,H_{kv}=2$，每两个 Q heads 共享一个 KV head。下面显式 repeat 只用于教学参考；生产 kernel 应广播读取而不复制。

In [ ]:
def expand_kv_for_reference(kv, query_heads):
    kv_heads = kv.shape[1]
    if query_heads % kv_heads:
        raise ValueError('query_heads 必须能被 kv_heads 整除')
    return np.repeat(kv, query_heads // kv_heads, axis=1)

Hq, Hkv, Dh = 4, 2, 3
Q_g = rng.normal(size=(1, Hq, L, Dh))
K_small = rng.normal(size=(1, Hkv, L, Dh))
V_small = rng.normal(size=(1, Hkv, L, Dh))
K_g, V_g = expand_kv_for_reference(K_small, Hq), expand_kv_for_reference(V_small, Hq)
O_g, _, _ = scaled_dot_attention(Q_g, K_g, V_g, mask4d)
print('small KV:', K_small.shape, 'logical expanded KV:', K_g.shape, 'output:', O_g.shape)
print('q->kv mapping:', np.arange(Hq) // (Hq // Hkv))

## 7. 在线 softmax：FlashAttention 的关键积木

分块不能先对每块单独 softmax 再拼接，因为各块分母不同。在线算法维护运行最大值 $m$ 和指数和 $l$，新块到来时重标定旧统计。FlashAttention 再同步重标定输出累积，从而不物化完整概率矩阵。

In [ ]:
def online_softmax_1d(scores, block_size=3):
    m, total = -np.inf, 0.0
    for start in range(0, len(scores), block_size):
        block = scores[start:start + block_size]
        block_max = np.max(block)
        new_m = max(m, block_max)
        total = np.exp(m - new_m) * total + np.exp(block - new_m).sum()
        m = new_m
    return np.exp(scores - m) / total

scores_1d = np.array([1000.0, 999.0, -1000.0, 1001.0, 997.0, 0.0])
reference = stable_softmax(scores_1d)
online = online_softmax_1d(scores_1d, block_size=2)
print('reference =', reference)
print('online    =', online)
print('max error =', np.max(np.abs(reference - online)))
assert np.allclose(reference, online)

## 8. 复杂度与简单 profile

QK 和 AV 为 $O(BL^2D)$，朴素概率矩阵为 $O(BHL^2)$ 元素；投影为 $O(BLD^2)$。Flash 改善 HBM IO 和激活存储，不改变全局 attention 的二次 FLOPs。下面只演示长度翻倍的趋势，NumPy 时间不是 GPU kernel 结论。

In [ ]:
for length in [64, 128, 256]:
    z = rng.normal(size=(length, 32))
    t0 = time.perf_counter()
    scaled_dot_attention(z, z, z)
    elapsed = (time.perf_counter() - t0) * 1000
    score_mib = length * length * 8 / 2**20  # 本实现 score 为 float64
    print(f'L={length:3d}: {elapsed:7.2f} ms, one score matrix={score_mib:.3f} MiB')

## 常见坑、工程变体与练习

- Softmax 沿 key 轴；布尔 True 是 keep 还是 mask 必须查 API；全 mask 行要安全处理。
- Transpose 后直接按错误 stride reshape 会混淆 head 和序列；生产中还要避免不必要 contiguous copy。
- Attention 权重只是当前头的路由系数，不是完整因果解释；还要看 V、输出投影、残差和干预结果。
- Flash 是 exact IO 优化；局部/稀疏减少 pair；线性 attention 改变核与模型函数。三者不能混为一谈。
- Prefill 是 $L\times L$，逐 token decode 是 $1\times L$；MHA/GQA/MQA 主要改变 KV 头存储与带宽，不让 Q heads 消失。

练习：①为 scaled_dot_attention 加 attention dropout；②实现不显式 repeat 的 GQA 循环参考；③对 mask 区做数值梯度泄漏检查；④计算 $D=4096,H_q=32,H_{kv}=8$ 的投影参数量。

面试主线：Q/K 匹配、V 读取 → shape 与 $\sqrt d$ → mask/稳定 softmax → 多头与 GQA → $L^2$ 成本 → Flash 的 tiling、在线 softmax与重计算。